In [1]:
%load_ext autoreload
%autoreload 2

import yaml
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

block_path = "../../experiment_data/balance_metrics/cross_layer_random.csv"
block_path_cifar = "../../experiment_data/balance_metrics/cross_layer_random_cifar.csv"

block_data = pd.concat([pd.read_csv(block_path), pd.read_csv(block_path_cifar)])

with open('./layer_orders_cross_layer.yml', 'r') as f:
    block_order = yaml.safe_load(f)
    
import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

block_data["rprop"] = block_data["dataset"].apply(lambda x: float(x.split("-r")[-1]))
block_data["dataset"] = block_data["dataset"].apply(lambda x: x.split("-r")[0])

In [36]:
densenet = block_data[(block_data['model']=='densenet') & (block_data['split']=='trainUval') & (block_data['dataset']=='cifar') & (block_data['rprop'].isin([0.0, 1.0]))].copy()

densenet["layer_idx"] = densenet["layer"].apply(lambda x: block_order['densenet'][x]['idx'])
densenet["layer_names"] = densenet["layer"].apply(lambda x: block_order['densenet'][x]['name'])
densenet.sort_values("layer_idx", inplace=True)

fig = px.line(densenet, x="layer_names", y="sackin_index", color="rprop", color_discrete_sequence=px.colors.qualitative.Plotly)
fig.update_xaxes(tickangle=80, title_text="Layer")
fig.update_yaxes(type="log", title_text="Sackin Index (log scale)")
fig.update_layout(showlegend=False, margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2)

In [3]:
resnet = block_data[(block_data['model']=='resnet') & (block_data['split']=='trainUval') & (block_data['dataset']=='cifar') & (block_data['rprop'].isin([0.0, 1.0]))].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: block_order['resnet'][x]['idx'])
resnet["layer_names"] = resnet["layer"].apply(lambda x: block_order['resnet'][x]['name'])
resnet.sort_values(["layer_idx", "rprop"], inplace=True)

fig = px.line(resnet, x="layer_names", y="sackin_index", color="rprop", color_discrete_sequence=px.colors.qualitative.Plotly)
fig.update_xaxes(tickangle=80, title_text="Layer")
fig.update_yaxes(type="log", title_text="Sackin Index (log scale)")
fig.update_layout(showlegend=False, margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2)

In [10]:
fig = px.line(resnet[resnet["rprop"] == 0.0], x="layer_names", y="sackin_index")

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Sackin Index", secondary_y=False)
fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=250*2, height=200*2, font=dict(size=14), showlegend=False)
# fig.show()
fig.write_image(f"interface-across-blocks.png", scale=8)